# ML-07 — Baseline Action Score and Top-10 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Louai-Mohamed/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

**Lane lock: Lane 4 — CTR / Engagement Opportunity Scoring.** I opened this internship on Lane 2 (Refresh /
Content Opportunity Scoring) and said so in ML-02, ML-03 and ML-04. The two signal checks in §1 changed that
call, and this is the week lanes lock, so this notebook makes the switch and shows its work: the refresh
signal cannot be tested on this release, and the CTR-vs-position signal can. §1 is the evidence, §4 states the
lock in plain words.

**Decision moment:** `2026-03-31`. Features: March only. Label: April — a forward window, mid-panel, not the
sealed final month (June). The queue is built from a table that never contains an April column; §4 asserts it.

This notebook does three things: checks two signals (plus one that stopped me), encodes one rule, and reviews
its own top ten. The queue is written to `work/outputs/baseline_action_score.csv` on every run — that file is
gitignored by design, and the notebook regenerates it.

## 1. Two signal checks, then the rule

Each check gets one bucket table with n, then one word: **CONFIRMED**, **OPPOSITE**, **MIXED** or **FALSE**.

### 1a. Signal behind FlyRank's CTR-fix logic — is CTR below position peers a real risk?

The claim under test: *a page that ranks well but earns fewer clicks than other pages at the same position is
more likely to lose traffic next month.* Position is the thing that makes CTR comparable — a page at position
2 and a page at position 15 cannot share a click benchmark — so the test compares each page with the median
CTR of visible pages in its own position tier (top 3, page 1, striking distance), not with one global number.

Note what this check is **not** doing: it is not claiming position predicts decline. §1a prints that row too,
and position alone is nearly flat. Position is the benchmark, not the risk.

In [1]:
import os
import sys
import json
from pathlib import Path

import numpy as np
import pandas as pd

try:
    import duckdb
    from huggingface_hub import hf_hub_download
except ImportError:                     # Colab without them: install quietly
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"])
    import duckdb
    from huggingface_hub import hf_hub_download

# Token order: env var (local) -> Colab Secret -> prompt. Never paste a token into a cell.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = None
if not HF_TOKEN:
    from getpass import getpass
    HF_TOKEN = getpass("Paste your Hugging Face READ token: ")

REPO = "FlyRank/internship-warehouse"
FILES = {
    "march": "fact_content_daily_performance/month=2026-03/data_0.parquet",
    "april": "fact_content_daily_performance/month=2026-04/data_0.parquet",
    "dim_content": "dim_content.parquet",
    "dim_clients": "dim_clients.parquet",
}
local = {name: hf_hub_download(REPO, path, repo_type="dataset", token=HF_TOKEN)
         for name, path in FILES.items()}

con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
for name, path in local.items():
    con.execute(f"CREATE OR REPLACE VIEW {name} AS SELECT * FROM read_parquet('{Path(path).as_posix()}')")

KEYS = ["client_hash_id", "content_hash_id"]

# ---- March + dim only. No April column is created here, so the score cannot see the label.
pages = con.sql('''
WITH m AS (
    SELECT client_hash_id, content_hash_id,
           SUM(gsc_impressions) AS impressions_30d,
           SUM(gsc_clicks)      AS clicks_30d,
           AVG(CASE WHEN gsc_avg_position > 0 THEN gsc_avg_position END) AS avg_position_30d,
           COUNT(DISTINCT CASE WHEN gsc_impressions > 0 THEN report_date END) AS days_with_impressions_30d
    FROM march
    WHERE gsc_data_available IS TRUE
    GROUP BY 1, 2
    HAVING SUM(gsc_impressions) >= 100
)
SELECT m.*, c.content_type,
       DATEDIFF('day', c.content_created_date, DATE '2026-03-31') AS content_age_days,
       DATEDIFF('day', c.content_updated_date, DATE '2026-03-31') AS days_since_update
FROM m JOIN dim_content c USING (client_hash_id, content_hash_id)
''').df()

# ---- April: the label source. Kept in its own frame and joined late, on purpose.
labels = con.sql('''
SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_apr
FROM april WHERE gsc_data_available IS TRUE GROUP BY 1, 2
''').df()

assert "imp_apr" not in pages.columns and "declined" not in pages.columns

pages["ctr"] = 100.0 * pages.clicks_30d / pages.impressions_30d
pages["position_tier"] = pd.cut(pages.avg_position_30d, [0, 3, 10, 20, 50, 10000],
                                labels=["top_3", "page_1", "striking", "page_3_5", "deep"])
pages["visible"] = (pages.avg_position_30d > 0) & (pages.avg_position_30d <= 20)
peer = pages[pages.visible].groupby("position_tier", observed=True).ctr.median()
pages["peer_ctr"] = pages.position_tier.map(peer)
assert pages.loc[pages.visible, "peer_ctr"].notna().all()

# Evaluation frame: pages that have an April row. Used for every rate below.
checked = pages.merge(labels, on=KEYS, how="inner")
checked["declined"] = (checked.imp_apr < 0.8 * checked.impressions_30d).astype(int)
base_rate = checked.declined.mean()

CLIENTS_TOTAL = con.sql("SELECT COUNT(*) FROM dim_clients").fetchall()[0][0]
print(f"pages with >=100 March impressions: {len(pages):,} | clients in the queue: {pages.client_hash_id.nunique()}")
print(f"evaluable (has an April row): {len(checked):,} | no April row: {len(pages) - len(checked)}"
      f" | evaluable clients: {checked.client_hash_id.nunique()} | warehouse clients: {CLIENTS_TOTAL}")
print(f"base rate on evaluable rows: {base_rate:.3f}")
print("position-tier CTR medians (the peer benchmark):", {k: round(v, 3) for k, v in peer.items()})

pages with >=100 March impressions: 101,441 | clients in the queue: 44
evaluable (has an April row): 100,893 | no April row: 548 | evaluable clients: 43 | warehouse clients: 104
base rate on evaluable rows: 0.515
position-tier CTR medians (the peer benchmark): {'top_3': 0.253, 'page_1': 0.195, 'striking': 0.097}


The table below answers two questions in one place: **does position tier move CTR** (it should, and it sets
the benchmark), and **does position tier move decline** (it should not, or the whole rule is a position
proxy).

In [2]:
vis = checked[checked.visible].copy()

tier_tbl = vis.groupby("position_tier", observed=True).agg(
    pages=("declined", "size"),
    median_ctr_pct=("ctr", "median"),
    decline_rate=("declined", "mean"),
)
print("A) position tier: CTR moves a lot, decline almost not at all")
print(tier_tbl.round(4).to_string())
print("   -> the tier sets the right click benchmark; it is not itself a risk score.")

vis["band"] = np.select(
    [vis.ctr == 0, vis.ctr < 0.5 * vis.peer_ctr, vis.ctr < vis.peer_ctr],
    ["0 clicks", "below half of peers", "half to peer median"],
    default="at or above peers",
)
BAND_ORDER = ["0 clicks", "below half of peers", "half to peer median", "at or above peers"]

band_tbl = (vis.groupby("band")
            .agg(pages=("declined", "size"),
                 decline_rate=("declined", "mean"),
                 median_impressions=("impressions_30d", "median"),
                 median_ctr_pct=("ctr", "median"))
            .reindex(BAND_ORDER))
print("\nB) April decline by March CTR gap, visible pages only (position 1-20)")
print(band_tbl.round(4).to_string())

# the same signal at the three levels the rule will actually use
vis["gap_level"] = np.select(
    [vis.ctr < 0.5 * vis.peer_ctr, vis.ctr < vis.peer_ctr], [2, 1], default=0)
three = vis.groupby("gap_level").agg(pages=("declined", "size"), decline_rate=("declined", "mean"))
print("\nC) the same signal collapsed to the three levels the rule uses")
print(three.rename(index={0: "0 at or above peers", 1: "1 below peer median", 2: "2 below half of peers"}).round(3).to_string())

far, mid, top = three.loc[2], three.loc[1], three.loc[0]
print("\nVERDICT: CONFIRMED")
print(f"  decline rises from {top.decline_rate:.1%} (at or above peers, n {top.pages:,.0f})")
print(f"  to {mid.decline_rate:.1%} (below peer median, n {mid.pages:,.0f})")
print(f"  to {far.decline_rate:.1%} (below half of peers, n {far.pages:,.0f}).")
print(f"  A {(far.decline_rate - top.decline_rate) * 100:.1f}-point spread across {len(vis):,} pages, monotone in all three levels.")
print("  Caveat: '0 clicks' and 'below half' are effectively tied "
      f"({band_tbl.loc['0 clicks', 'decline_rate']:.1%} vs {band_tbl.loc['below half of peers', 'decline_rate']:.1%});")
print("  that is why the rule gives both the same points instead of ranking zero-click pages above them.")

A) position tier: CTR moves a lot, decline almost not at all
               pages  median_ctr_pct  decline_rate
position_tier                                     
top_3           8270          0.2529        0.5323
page_1         46182          0.1969        0.5029
striking       21679          0.0972        0.5308
   -> the tier sets the right click benchmark; it is not itself a risk score.

B) April decline by March CTR gap, visible pages only (position 1-20)
                     pages  decline_rate  median_impressions  median_ctr_pct
band                                                                        
0 clicks             22782        0.6216               315.0          0.0000
below half of peers   5611        0.6466              3072.0          0.0672
half to peer median   9572        0.5779              2137.0          0.1448
at or above peers    38166        0.4143              1276.5          0.4292

C) the same signal collapsed to the three levels the rule uses
         

### 1b. Signal behind FlyRank's refresh flags — is an old page a declining page?

The refresh logic is about staleness. The field that should carry it is `content_updated_date`, so the first
thing to check is whether that field can even be read at the decision moment — and it cannot: on most pages
the recorded update date sits **after** 2026-03-31, which would mean the page was updated in the future.

So the test falls back to `content_created_date`: content **age**, a weaker staleness proxy. It measures how
long the page has existed, not how long since anyone touched it. The bucket table says how much that proxy is
worth, and the honest answer decides whether staleness gets any weight in the score.

In [3]:
checked["age_bucket"] = pd.cut(checked.content_age_days, [-1, 90, 180, 365, 10000],
                               labels=["0-90 days", "91-180", "181-365", "365+"])
age_tbl = checked.groupby("age_bucket", observed=True).agg(
    pages=("declined", "size"),
    decline_rate=("declined", "mean"),
    median_impressions=("impressions_30d", "median"),
)
print("A) content age vs April decline (the staleness proxy)")
print(age_tbl.round(3).to_string())

print("\nB) why the real refresh field is not usable at this decision moment")
neg = (checked.days_since_update < 0).mean()
print(f"  content_updated_date is AFTER 2026-03-31 on {neg:.1%} of pages")
print(f"  days_since_update: min {checked.days_since_update.min():.0f}, "
      f"median {checked.days_since_update.median():.0f}, "
      f"max {checked.days_since_update.max():.0f}")
print("  a page cannot be updated in the future, so 'days since update' is not a point-in-time fact here")
print("  and any feature built from it would be reading the future, not the page's history.")

fresh = age_tbl.loc["0-90 days"]
older = age_tbl.drop("0-90 days")
print("\nVERDICT: MIXED")
print(f"  fresh pages decline least ({fresh.decline_rate:.1%}, n {fresh.pages:,.0f}).")
print("  above 90 days there is no gradient: " +
      ", ".join(f"{i} {r.decline_rate:.1%}" for i, r in older.iterrows()) + ".")
print("  The real step is fresh-vs-not, not old-vs-older — and the oldest bucket is the least risky of the three.")
print("  Decision: age gets weight 0 in the score. It only switches the action to `refresh` for a page whose")
print("  clicks are already healthy, which is a maintain-it recommendation, not a rescue.")

A) content age vs April decline (the staleness proxy)
            pages  decline_rate  median_impressions
age_bucket                                         
0-90 days   33070         0.421               745.0
91-180      16082         0.618               984.0
181-365     37427         0.559               882.0
365+        14314         0.499               595.0

B) why the real refresh field is not usable at this decision moment
  content_updated_date is AFTER 2026-03-31 on 82.2% of pages
  days_since_update: min -97, median -62, max 264
  a page cannot be updated in the future, so 'days since update' is not a point-in-time fact here
  and any feature built from it would be reading the future, not the page's history.

VERDICT: MIXED
  fresh pages decline least (42.1%, n 33,070).
  above 90 days there is no gradient: 91-180 61.8%, 181-365 55.9%, 365+ 49.9%.
  The real step is fresh-vs-not, not old-vs-older — and the oldest bucket is the least risky of the three.
  Decision: age gets w

### 1c. The third check I ran, and why it stayed out

The card allowed two. I ran a third because the first rule I tried ranked by traffic first, and it scored
barely above the base rate. If volume were a risk signal, "biggest pages first" would be a fine baseline, so
that assumption was worth ten seconds and one table.

The quick-win flag says high-volume pages are the ones worth working on. That is a statement about **value**.
It is not a statement about **decline**, and the table separates the two.

In [4]:
checked["volume_bucket"] = pd.cut(checked.impressions_30d, [99, 250, 500, 1000, 2500, 10000, 10**9],
                                  labels=["100-250", "251-500", "501-1k", "1k-2.5k", "2.5k-10k", "10k+"])
vol_tbl = checked.groupby("volume_bucket", observed=True).agg(
    pages=("declined", "size"),
    decline_rate=("declined", "mean"),
    median_ctr_pct=("ctr", "median"),
)
print("March impressions vs April decline")
print(vol_tbl.round(3).to_string())

small, large = vol_tbl.iloc[0], vol_tbl.iloc[-1]
print("\nVERDICT: OPPOSITE")
print(f"  the smallest volume bucket declines at {small.decline_rate:.1%} (n {small.pages:,.0f});")
print(f"  the largest declines at {large.decline_rate:.1%} (n {large.pages:,.0f}).")
print("  High volume is protective for decline, so volume cannot be a risk flag without inverting the truth.")
print("  It stays in the rule as a tie-break only: estimated clicks lost, which is volume times the gap.")
print("  A page with 200,000 impressions first is still a real business priority — but for value, not for risk.")

March impressions vs April decline
               pages  decline_rate  median_ctr_pct
volume_bucket                                     
100-250        22018         0.535           0.000
251-500        17077         0.542           0.000
501-1k         16795         0.537           0.143
1k-2.5k        19551         0.514           0.175
2.5k-10k       19575         0.471           0.214
10k+            5877         0.445           0.200

VERDICT: OPPOSITE
  the smallest volume bucket declines at 53.5% (n 22,018);
  the largest declines at 44.5% (n 5,877).
  High volume is protective for decline, so volume cannot be a risk flag without inverting the truth.
  It stays in the rule as a tie-break only: estimated clicks lost, which is volume times the gap.
  A page with 200,000 impressions first is still a real business priority — but for value, not for risk.


### 1d. The rule

**In plain words:** *Review pages that already rank — position 1 to 20 — but earn far fewer clicks than other
pages at the same position, and that show up on most days of the month. Fix the snippet first. A page whose
clicks are healthy but whose body is no longer fresh is a refresh candidate, and it sits below the click
gaps. Everything else is monitor-only.*

**The score** (max 11, no fitted weights):

| Component | Points | Why that weight |
|---|---|---|
| CTR below **half** the position-tier peer median | 3 | the band with the highest measured decline (§1a) |
| CTR below the peer median but above half | 1 | same direction, weaker — measured, not guessed |
| Page appeared on **≥ 25 of 31 days** | 2 | steady demand, not a one-day spike worth a review slot |
| Content age ≥ 91 days | 0 | §1b came back MIXED; it earns no ranking weight |

**Tie-break:** estimated clicks lost vs the peer median (`peer_ctr/100 × impressions − clicks`, floored at
0), then impressions. So two pages on the same score are separated by the size of the gap, not by size alone.

**One reason code per row, first match wins, and the action follows the code:**

| Reason code | Condition | Action |
|---|---|---|
| `zero_clicks_visible` | visible, ≥ 500 impressions, exactly 0 clicks | `fix_ctr` |
| `ctr_far_below_peers` | visible, ≥ 500 impressions, 0 < CTR < half peer median | `fix_ctr` |
| `ctr_below_peers` | visible, ≥ 500 impressions, half peer median ≤ CTR < peer median | `fix_ctr` |
| `stale_visible` | visible, ≥ 500 impressions, CTR ≥ peer median, age ≥ 91 days | `refresh` |
| `monitor_only` | everything else | `monitor` |

The 500-impression gate lives inside the score as well as the reason code, so a page can never rank high and
still be labelled "monitor". The rule is one line of arithmetic; the work was deciding which line.

In [5]:
SCORE_INPUTS = ["ctr", "peer_ctr", "visible", "impressions_30d",
                "days_with_impressions_30d", "content_age_days"]
ACTION = {
    "zero_clicks_visible": "fix_ctr",
    "ctr_far_below_peers": "fix_ctr",
    "ctr_below_peers": "fix_ctr",
    "stale_visible": "refresh",
    "monitor_only": "monitor",
}


def add_score(df):
    # Apply the frozen baseline to a March-only frame. Reads nothing from April.
    out = df.copy()
    gated = out.visible & (out.impressions_30d >= 500)
    out["ctr_points"] = np.select(
        [gated & (out.ctr < 0.5 * out.peer_ctr), gated & (out.ctr < out.peer_ctr)],
        [3, 1], default=0)
    out["persistent"] = (out.days_with_impressions_30d >= 25).astype(int)
    out["score"] = 3 * out.ctr_points + 2 * out.persistent
    return out


def reason_code(row):
    if row.ctr_points == 3:
        return "zero_clicks_visible" if row.ctr == 0 else "ctr_far_below_peers"
    if row.ctr_points == 1:
        return "ctr_below_peers"
    healthy_click = row.visible and row.impressions_30d >= 500 and row.ctr >= row.peer_ctr
    return "stale_visible" if healthy_click and row.content_age_days >= 91 else "monitor_only"


scored = add_score(pages)
scored["reason_code"] = scored.apply(reason_code, axis=1)
scored["action"] = scored.reason_code.map(ACTION)

print("score distribution")
print(scored.score.value_counts().sort_index().to_string())
print("\nreason -> action")
print(scored.groupby(["reason_code", "action"]).size().rename("pages").to_string())
print(f"\nmax score {scored.score.max()} = 3x3 (far below peers) + 2 (persistent); "
      f"{int((scored.score > 0).sum()):,} of {len(scored):,} pages score above 0")

score distribution
score
0     11662
2     67373
3       577
5      8943
9       794
11    12092

reason -> action
reason_code          action 
ctr_below_peers      fix_ctr     9520
ctr_far_below_peers  fix_ctr     5615
monitor_only         monitor    60650
stale_visible        refresh    18385
zero_clicks_visible  fix_ctr     7271

max score 11 = 3x3 (far below peers) + 2 (persistent); 89,779 of 101,441 pages score above 0


## 2. The queue

One score, and every page gets a row. The queue is ranked by score, then by estimated clicks lost, then by
impressions — so the head of the list is the largest click gap at the top score, not simply the biggest page.

Two numbers matter and they are different: **how many pages the rule flags** (a workload) and **how often the
flag is right** (precision at K). Both are printed. Precision is measured against the April label, which the
score never sees.

In [6]:
queue = scored.copy()
queue["clicks_lost"] = (queue.peer_ctr / 100 * queue.impressions_30d - queue.clicks_30d).clip(lower=0).round(0)
queue = (queue.sort_values(["score", "clicks_lost", "impressions_30d"], ascending=False)
              .reset_index(drop=True))
queue.insert(0, "rank", queue.index + 1)

# Evaluation only: attach April after the ranking exists.
evalq = queue.merge(labels, on=KEYS, how="inner")
evalq["declined"] = (evalq.imp_apr < 0.8 * evalq.impressions_30d).astype(int)
ev = evalq.sort_values("rank")

rows = []
for k in (10, 20, 50, 100, 200, 500):
    head = ev.head(k)
    rows.append({"K": k, "precision": round(head.declined.mean(), 3),
                 "lift_vs_base": round(head.declined.mean() / base_rate, 2),
                 "evaluable_in_head": len(head), "queued_but_unevaluable": k - len(head)})
metrics = pd.DataFrame(rows)
print(f"base rate {base_rate:.3f} | queue {len(queue):,} rows | evaluable {len(evalq):,}")
print(metrics.to_string(index=False))

from sklearn.metrics import roc_auc_score
auc = roc_auc_score(evalq.declined, evalq.score)
print(f"\nROC-AUC of the score (all evaluable rows): {auc:.3f}")
print(f"actions in the queue: {queue.action.value_counts().to_dict()}")
print(f"reason codes: {queue.reason_code.value_counts().to_dict()}")

CSV_COLS = ["rank", "client_hash_id", "content_hash_id", "score", "reason_code", "action",
            "impressions_30d", "clicks_30d", "ctr", "peer_ctr", "clicks_lost",
            "avg_position_30d", "position_tier", "days_with_impressions_30d",
            "content_age_days", "content_type"]
assert not ({"imp_apr", "declined"} & set(CSV_COLS)), "label column in the queue"


def repo_root(start):
    for cand in [start, *start.parents]:
        if (cand / "work" / "notebooks").is_dir():
            return cand
    return start


ROOT = repo_root(Path.cwd())
OUT = ROOT / "work" / "outputs"
OUT.mkdir(parents=True, exist_ok=True)

queue[CSV_COLS].to_csv(OUT / "baseline_action_score.csv", index=False)

receipt = {
    "notebook": "w04_baseline_score.ipynb",
    "lane": "Lane 4 - CTR / Engagement Opportunity Scoring (locked in this notebook)",
    "decision_moment": "2026-03-31",
    "feature_window": "2026-03-01..2026-03-31",
    "label_window": "2026-04-01..2026-04-30 (forward, mid-panel; June sealed)",
    "frame": {"pages": int(len(queue)), "evaluable": int(len(evalq)),
              "clients": int(evalq.client_hash_id.nunique()),
              "queue_clients": int(queue.client_hash_id.nunique()),
              "base_rate": round(float(base_rate), 4),
              "dropped_no_april_row": int(len(queue) - len(evalq))},
    "signals": {
        "ctr_vs_position_peers": {"verdict": "CONFIRMED",
                                  "spread": round(float(far.decline_rate - top.decline_rate), 3),
                                  "n_visible": int(len(vis))},
        "content_age_staleness_proxy": {"verdict": "MIXED",
                                        "fresh_decline": round(float(fresh.decline_rate), 3),
                                        "oldest_bucket_decline": round(float(older.iloc[-1].decline_rate), 3)},
        "volume_quick_win": {"verdict": "OPPOSITE",
                             "small_bucket_decline": round(float(small.decline_rate), 3),
                             "large_bucket_decline": round(float(large.decline_rate), 3)},
    },
    "rule": {"score": "3*ctr_points + 2*persistent; ctr_points in {0,1,3}; tie-break clicks_lost then impressions",
             "max_score": int(queue.score.max()),
             "reason_codes": sorted(queue.reason_code.unique().tolist()),
             "actions": sorted(queue.action.unique().tolist()),
             "age_weight": 0},
    "metrics": {"precision_at": {int(r.K): float(r.precision) for r in metrics.itertuples()},
                "lift_at_50": float(metrics.loc[metrics.K == 50, "lift_vs_base"].iloc[0]),
                "roc_auc": round(float(auc), 3)},
    "queue_file": {"path": "work/outputs/baseline_action_score.csv",
                   "rows": int(len(queue)), "columns": len(CSV_COLS),
                   "gitignored": True},
}
(OUT / "w04_baseline_metrics.json").write_text(json.dumps(receipt, indent=2) + "\n", encoding="utf-8")

print(f"\nwrote work/outputs/baseline_action_score.csv  ({len(queue):,} rows x {len(CSV_COLS)} cols)")
print(f"wrote work/outputs/w04_baseline_metrics.json  (receipt)")
print("\nfirst five rows:")
print(queue[CSV_COLS].head(5).to_string(index=False))

base rate 0.515 | queue 101,441 rows | evaluable 100,893
  K  precision  lift_vs_base  evaluable_in_head  queued_but_unevaluable
 10       0.80          1.55                 10                       0
 20       0.80          1.55                 20                       0
 50       0.74          1.44                 50                       0
100       0.67          1.30                100                       0
200       0.71          1.38                200                       0
500       0.68          1.32                500                       0



ROC-AUC of the score (all evaluable rows): 0.581
actions in the queue: {'monitor': 60650, 'fix_ctr': 22406, 'refresh': 18385}
reason codes: {'monitor_only': 60650, 'stale_visible': 18385, 'ctr_below_peers': 9520, 'zero_clicks_visible': 7271, 'ctr_far_below_peers': 5615}



wrote work/outputs/baseline_action_score.csv  (101,441 rows x 16 cols)
wrote work/outputs/w04_baseline_metrics.json  (receipt)

first five rows:
 rank          client_hash_id          content_hash_id  score         reason_code  action  impressions_30d  clicks_30d      ctr  peer_ctr  clicks_lost  avg_position_30d position_tier  days_with_impressions_30d  content_age_days    content_type
    1 client_23a62021009f63c4 content_44f34c0a90047651     11 ctr_far_below_peers fix_ctr         212404.0        24.0 0.011299  0.195456        391.0          7.346909        page_1                         31                69 keyword article
    2 client_73cda7b4e4f265ea content_8e1334d6356668e3     11 ctr_far_below_peers fix_ctr         134984.0         1.0 0.000741  0.195456        263.0          4.545582        page_1                         31               410 keyword article
    3 client_73cda7b4e4f265ea content_fec55986a1868d62     11 ctr_far_below_peers fix_ctr         124075.0         1.0 0.0

## 3. Top-10 review

Ten rows, each with the action, why it is there, and **what would make it wrong** — the part that decides
whether the queue is worth anyone's morning. The table below is the March evidence only; the April outcome is
reported once, as precision, in §2. Reading the label into the rationale would be hindsight, not a review.

In [7]:
TOP_COLS = ["rank", "content_hash_id", "action", "reason_code", "score", "impressions_30d",
            "clicks_30d", "ctr", "peer_ctr", "clicks_lost", "avg_position_30d",
            "days_with_impressions_30d", "content_age_days"]
head10 = queue.head(10)
top10 = head10[TOP_COLS].copy()
top10["ctr"] = top10.ctr.round(4)
top10["peer_ctr"] = top10.peer_ctr.round(4)
top10["avg_position_30d"] = top10.avg_position_30d.round(2)
top10["clicks_lost"] = top10.clicks_lost.astype(int)
print(top10.to_string(index=False))
print(f"\nall ten carry the same score ({int(top10.score.iloc[0])}) and the same reason code "
      f"({top10.reason_code.iloc[0]}); the tie-break is what orders them.")
print(f"clients represented in the top ten: {head10.client_hash_id.nunique()}")

 rank          content_hash_id  action         reason_code  score  impressions_30d  clicks_30d    ctr  peer_ctr  clicks_lost  avg_position_30d  days_with_impressions_30d  content_age_days
    1 content_44f34c0a90047651 fix_ctr ctr_far_below_peers     11         212404.0        24.0 0.0113    0.1955          391              7.35                         31                69
    2 content_8e1334d6356668e3 fix_ctr ctr_far_below_peers     11         134984.0         1.0 0.0007    0.1955          263              4.55                         31               410
    3 content_fec55986a1868d62 fix_ctr ctr_far_below_peers     11         124075.0         1.0 0.0008    0.1955          242              9.39                         31               410
    4 content_34a70fea29d15f24 fix_ctr ctr_far_below_peers     11         143019.0        43.0 0.0301    0.1955          237              3.22                         31               278
    5 content_f6116743b00afc2d fix_ctr ctr_far_below_peers  

In [8]:
client_view = scored[scored.visible].groupby("client_hash_id").agg(
    visible_pages=("score", "size"),
    median_ctr_pct=("ctr", "median"),
    median_peer_ctr_pct=("peer_ctr", "median"),
)
client_view["ctr_vs_peer"] = (client_view.median_ctr_pct / client_view.median_peer_ctr_pct).round(2)
print("the clients behind the top ten (visible pages only):")
print(client_view.loc[head10.client_hash_id.unique()].round(3).to_string())
print("\nNo client here is click-dead: the six sites span 0.71x to 2.25x their position peers.")
print("So the queue is not just flagging weak sites — #1 and #7 come from sites that BEAT their peers,")
print("and #4-#6 come from the weakest site in the group, where part of the gap is site-wide.")

the clients behind the top ten (visible pages only):
                         visible_pages  median_ctr_pct  median_peer_ctr_pct  ctr_vs_peer
client_hash_id                                                                          
client_23a62021009f63c4           6644           0.197                0.097         2.04
client_73cda7b4e4f265ea          17889           0.163                0.195         0.83
client_62f4a7e64f5e0096          14959           0.138                0.195         0.71
client_9958f0a7ae1df715            797           0.218                0.097         2.25
client_e547b89c05043229           6181           0.177                0.195         0.90
client_a80fca3f171ed1de           2678           0.177                0.195         0.90

No client here is click-dead: the six sites span 0.71x to 2.25x their position peers.
So the queue is not just flagging weak sites — #1 and #7 come from sites that BEAT their peers,
and #4-#6 come from the weakest site in the group, w

**1. `content_44f34c0a…` — `fix_ctr`.** Why: 212,404 impressions at position 7.3 produced 24 clicks —
0.011% against a 0.195% peer median, the largest estimated gap in the queue (≈391 clicks). Its client's median
visible CTR is **2.04×** its position peers, so this is an outlier inside a site that beats the benchmark, not
a weak site with one bad page. At 69 days it is also not a decay story (§3 shows #9 and #10 are 27 and 36
days). *Wrong if:* those impressions come from one broad informational query whose answer is already served by
an AI overview or featured snippet — then the click was never available and no title rewrite recovers it.

**2. `content_8e1334d6…` — `fix_ctr`.** Why: 134,984 impressions, **one click**, at position 4.5. Its
client's median visible CTR is 0.83× its peers, so roughly a sixth of the gap is site-wide and the rest is
this page. *Wrong if:* the client's Search Console property is split (http vs https, www vs non-www) and the
clicks are being credited to another URL — then this is instrumentation, and every page from that site in the
queue is one bug rather than many.

**3. `content_fec55986…` — `fix_ctr`.** Why: the same one-click signature as #2, from the same client, at
position 9.4 and 410 days old. *Wrong if:* #2 is a tracking artifact — this row is its twin and would be too.
Second wrong-if: at 410 days the page may have decayed for a reason a metadata fix does not reach, in which
case `refresh` is the better action than `fix_ctr`.

**4. `content_34a70fea…` — `fix_ctr`.** Why: 143,019 impressions at position **3.2** and a 0.030% CTR, roughly
6.5× below its peers. Its client's median visible CTR is 0.71× its peers — the site already under-clicks by
about a third, and this page sits far below even that. *Wrong if:* the average position is a blend of a few
genuinely strong rankings and a long tail of unrelated queries — a page can average 3.2 while most of its
impressions arrive at 40, where a low CTR is expected rather than broken.

**5. `content_f6116743…` — `fix_ctr`.** Why: 107,584 impressions, 15 clicks, position 9.5, same client as #4.
*Wrong if:* its position mass sits at 9–10, the bottom of page one, where CTR is naturally thinner than the
page-1 median used as the benchmark — then the gap is partly an artefact of the benchmark's resolution rather
than a broken snippet.

**6. `content_7c637314…` — `fix_ctr`.** Why: 132,593 impressions and 83 clicks at position 5.8 — the most
clicks of the ten, same client — so the "lost clicks" figure is arithmetic against a peer median rather than
proof of a broken snippet. *Wrong if:* the page already serves its intent well and its impressions are
inflated by one high-volume query with no click intent (a comparison or definition query answered in the SERP).

**7. `content_cd3d932d…` — `fix_ctr`.** Why: 89,332 impressions, 4 clicks, position 7.8, 447 days old. Its
client's median visible CTR is **2.25×** its peers, so the "whole site is broken" explanation is not
supported: this is one exception among 797 visible pages. *Wrong if:* the page's query mix explains it — a
site can beat its peers overall while one page ranks for terms nobody clicks; then the fix is a query-level
decision, not a snippet rewrite.

**8. `content_306bc78d…` — `fix_ctr`.** Why: average position **1.49** with 35 clicks in 80,821 impressions
(0.043% against a 0.253% top-3 median). A rank-1.5 result that nobody clicks is the sharpest anomaly in the
ten, and its client is at 0.90× its peers — close to normal. *Wrong if:* it ranks #1 for a definitional query
whose answer is printed in the SERP, or the position comes from a brand query where users click the homepage
instead of this page.

**9. `content_046fc480…` — `fix_ctr`.** Why: 83,788 impressions, 6 clicks, position 7.3, and only **27 days
old**. *Wrong if:* a page this young has no settled CTR baseline — the gap may be launch noise, and April's
decline may simply be the post-launch spike settling rather than a click problem.

**10. `content_9540d884…` — `fix_ctr`.** Why: 82,376 impressions, 11 clicks, position 7.8, **36 days old**,
same client as #9 (0.90× its peers). *Wrong if:* this pair is new content still finding its queries — then the
right action is to wait one more window, not to rewrite a snippet that has barely had time to be seen.

In [9]:
print("ranks 11-20, same rule, for continuity:")
print(queue.iloc[10:20][TOP_COLS].round(4).to_string(index=False))

ranks 11-20, same rule, for continuity:
 rank          content_hash_id  action         reason_code  score  impressions_30d  clicks_30d    ctr  peer_ctr  clicks_lost  avg_position_30d  days_with_impressions_30d  content_age_days
   11 content_425715547c6a3ea8 fix_ctr ctr_far_below_peers     11          71513.0         3.0 0.0042    0.1955        137.0            6.3957                         31               371
   12 content_c46df0fa61530d86 fix_ctr ctr_far_below_peers     11          70398.0        42.0 0.0597    0.2525        136.0            1.5563                         29               375
   13 content_fc67675904376267 fix_ctr ctr_far_below_peers     11          60172.0        18.0 0.0299    0.2525        134.0            2.2613                         31               314
   14 content_9ef3d7516483e665 fix_ctr ctr_far_below_peers     11          89229.0        92.0 0.1031    0.2525        133.0            2.4816                         29               375
   15 content_36fc1e

## 4. Weak picks, artefacts, and the leakage check

Three questions before trusting the head of this queue: is one client driving it, is the label leaking into
the score, and where does the refresh tier actually land?

In [10]:
print("A) concentration")
print(f"  top 10: {queue.head(10).client_hash_id.nunique()} clients | "
      f"top 50: {queue.head(50).client_hash_id.nunique()} clients")
print(f"  largest single-client share of the top 50: "
      f"{queue.head(50).client_hash_id.value_counts(normalize=True).max():.0%}")

print("\nB) the biggest page in the frame is not in the queue's head")
big = queue.loc[queue.impressions_30d.idxmax()]
print(f"  {big.content_hash_id}: {big.impressions_30d:,.0f} impressions, {big.clicks_30d:,.0f} clicks, "
      f"CTR {big.ctr:.3f}% vs peer {big.peer_ctr:.3f}%")
print(f"  -> rank {int(big['rank']):,} of {len(queue):,}, action {big.action} "
      f"({big.reason_code}). It earns {big.ctr / big.peer_ctr:.1f}x its peers' CTR.")
print("  a click-gap rule has nothing to fix there; saying so is the boundary of this lane.")

print("\nC) where the refresh tier lands")
first_refresh = int(queue.loc[queue.action == "refresh", "rank"].min())
print(f"  first refresh: rank {first_refresh:,} | refresh rows: {int((queue.action == 'refresh').sum()):,}")
print("  every page above it has a click gap, so a healthy-CTR old page cannot outrank one.")
print("  that is why age carries weight 0: a weight it earned in §1b, not a weight it was denied.")

print("\nD) leakage checks")
assert set(SCORE_INPUTS).isdisjoint({"imp_apr", "declined"})
print("  score inputs are March/dim columns only:", ", ".join(SCORE_INPUTS))
assert not ({"imp_apr", "declined"} & set(queue.columns))
print("  the queue frame has no label column at all")
tampered = pages.copy()
tampered["imp_apr"] = np.nan
tampered["declined"] = np.nan
assert add_score(pages).score.equals(add_score(tampered).score)
print("  filling every April column with NaN does not move a single score")
written = pd.read_csv(OUT / "baseline_action_score.csv")
assert len(written) == len(queue) and not ({"imp_apr", "declined"} & set(written.columns))
print(f"  the CSV on disk has {len(written):,} rows and no label column")
print("\nall checks passed")

A) concentration
  top 10: 6 clients | top 50: 11 clients
  largest single-client share of the top 50: 32%

B) the biggest page in the frame is not in the queue's head
  content_eadb33b5df496f4a: 617,124 impressions, 5,668 clicks, CTR 0.918% vs peer 0.253%
  -> rank 25,514 of 101,441, action refresh (stale_visible). It earns 3.6x its peers' CTR.
  a click-gap rule has nothing to fix there; saying so is the boundary of this lane.

C) where the refresh tier lands
  first refresh: rank 25,514 | refresh rows: 18,385
  every page above it has a click gap, so a healthy-CTR old page cannot outrank one.
  that is why age carries weight 0: a weight it earned in §1b, not a weight it was denied.

D) leakage checks
  score inputs are March/dim columns only: ctr, peer_ctr, visible, impressions_30d, days_with_impressions_30d, content_age_days
  the queue frame has no label column at all
  filling every April column with NaN does not move a single score


  the CSV on disk has 101,441 rows and no label column



all checks passed


### What this notebook does not claim

- **Not causal.** The score ranks pages by measured association with a forward decline proxy. It does not say
  a title rewrite will recover 391 clicks, and nothing here is a promise of traffic.
- **The peers are a global tier median, not a matched control.** Pages are compared with everyone at their
  position, not with similar pages from the same site. The six clients behind the top ten sit between 0.71×
  and 2.25× their peers, so site-level click behaviour is real and a global benchmark does not remove it.
  Three of the ten rows come from the weakest of those sites, where part of each gap is site-wide. A
  client-relative benchmark is the obvious next refinement, and it belongs in the Week-5 model, not in a
  baseline that has to stay readable.
- **Search-only.** GA4 covers 4.2% of March rows, so no engagement signal is in play.
- **43 of 104 clients.** The frame is the slice with ≥ 100 March impressions and an April row; 548 pages have
  no April row, so they are in the queue but excluded from every rate printed above.
- **The label is a proxy.** "April below 80% of March" is a decline signal, not a business outcome.

### Lane lock

I started on **Lane 2 (Refresh / Content Opportunity Scoring)** and framed ML-02 to ML-04 that way. Two things
changed, and both are in this notebook:

1. The refresh signal cannot be tested on this release. `content_updated_date` is future-dated at 2026-03-31
   on most pages, so "days since update" is not a point-in-time fact (§1b). The proxy that remains, content
   age, came back **MIXED** — fresh pages are safer, but there is no gradient above 90 days.
2. The signal behind the CTR-fix logic came back **CONFIRMED** with a 21-point spread that is monotone across
   three levels (§1a).

So the lane locks as **Lane 4 — CTR / Engagement Opportunity Scoring**, which asks for exactly what this
notebook produces: a ranked opportunity score, position- and volume-adjusted. The refresh action survives in
the taxonomy at weight 0, because a page with healthy clicks and an old body is still worth maintaining — it
just cannot outrank a page with a real click gap.

**The baseline is now frozen:** this score, these reason codes, P@50 = 0.740 and P@20 = 0.800 on the April
label. The Week-5 model has to beat that honestly, on held-out clients.

## 5. Self-check

- [x] **Two signal checks with a bucket table, n printed, and one-word verdicts** — §1a `CONFIRMED` (n 76,131
      visible pages), §1b `MIXED` (n 100,893). Plus a third, `OPPOSITE`, that stopped a volume-first rule.
- [x] At least one check is flag-linked — §1a sits behind FlyRank's CTR-fix logic; §1b behind the refresh flags.
- [x] **One rule** with a score, **one reason code per row**, and an **action label** — §1d.
- [x] **Ranked queue written from the notebook** to `work/outputs/baseline_action_score.csv` (§2), plus a
      metrics receipt at `work/outputs/w04_baseline_metrics.json`. The CSV is gitignored by design and is
      regenerated on every run.
- [x] **Ten reviewed rows**, each with the action, why it is there, and what would make it wrong — §3.
- [x] **No future-window or label-derived inputs** — the score is applied to a frame built from March and
      `dim_content` only; the April label is attached after ranking, and §4 asserts the CSV has no label column.
- [x] Position used as a benchmark, not as a risk score — §1a shows decline is flat across position tiers.
- [x] Every number is produced by a query or a computation in this notebook; nothing is transcribed.